In [ ]:
# In[1]:

import json

# Load the task.json file to understand the alert context
task_file_path = 'rca100/cases/t090/task.json'

with open(task_file_path, 'r') as file:
    task_data = json.load(file)

alert_title = task_data.get('alert_title')
alert_window = task_data.get('alert_window')
alert_entity = task_data.get('alert_entity')
prompt_text = task_data.get('prompt_text')

alert_title, alert_window, alert_entity, prompt_text

```
Out[1]:
```
The alert is titled "product-catalog 流量下跌告警" (product-catalog traffic drop alert). It occurred between 2026-04-12 21:26:08 and 2026-04-12 21:36:32 (UTC+8). The entity that triggered the alert is the operation `product-catalog::oteldemo.ProductCatalogService/ListProducts` within the APM domain. The user is requesting help to analyze the root cause of this issue.

The original code execution output of IPython Kernel is also provided below for reference:

('product-catalog 流量下跌告警', {'start': '2026-04-12T21:26:08+08:00', 'end': '2026-04-12T21:36:32+08:00'}, {'entity_id': '455f921f0afc16175724d9f27faf18e4', 'entity_name': 'product-catalog::oteldemo.ProductCatalogService/ListProducts', 'entity_type': 'apm.operation', 'entity_domain': 'apm'}, '<code vibeops_object type="alert_event">product-catalog 流量下跌告警<alert_event event_id="62bdd34ccb277ea3a2a6c17384cf2c8c" trans_id="mc4bf4bdm5203588dneo5j9ou4" rule_id="e876641e-1e5e-49d4-9e38-9233255b5baf" rule_name="product-catalog 流量下跌告警" /></code>\n\n帮我分析下根因。')```
```

In [ ]:
# In[2]:

import pandas as pd

# Load the metrics.parquet file
metrics_file_path = 'rca100/cases/t090/metrics.parquet'
metrics_df = pd.read_parquet(metrics_file_path)

# List all available KPIs and their corresponding entity sets
kpi_entity_sets = metrics_df[['metric', 'entity_set']].drop_duplicates()

kpi_entity_sets

```
Out[2]:
```
The `metrics.parquet` file contains 63 unique KPIs across various entity sets. Some examples of these KPIs include `node_ready_status`, `node_not_ready_status`, `node_memory_usage_rate`, and `node_disk_usage_rate` under the `k8s.node` entity set, as well as `workload`, `slow`, and `error` under the `apm.service.legacy` entity set. Additionally, the `error_rate` KPI is available under both `apm.operation` and `apm.instance` entity sets. For a comprehensive view, you can request to display more rows of the data.

The original code execution output of IPython Kernel is also provided below for reference:

metric          entity_set
0           node_ready_status            k8s.node
2175    node_not_ready_status            k8s.node
3625   node_memory_usage_rate            k8s.node
6380     node_disk_usage_rate            k8s.node
7830   node_pod_running_count            k8s.node
...                       ...                 ...
80815                workload  apm.service.legacy
80827                    slow  apm.service.legacy
80843                   error  apm.service.legacy
91107              error_rate       apm.operation
91113              error_rate        apm.instance

[63 rows x 2 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

# Filter the metrics data for the specific entity and entity set
filtered_metrics = metrics_df[
    (metrics_df['entity_set'] == 'apm.operation') &
    (metrics_df['entity_name'] == 'product-catalog::oteldemo.ProductCatalogService/ListProducts') &
    (metrics_df['metric'] == 'workload')
]

# Ensure the 'value' column is numeric for calculations
filtered_metrics['value'] = pd.to_numeric(filtered_metrics['value'], errors='coerce')

# Calculate the global P95 threshold for the 'workload' KPI
p95_threshold = filtered_metrics['value'].quantile(0.95)

p95_threshold

```
Out[3]:
```
The global P95 threshold for the `workload` KPI could not be calculated because the `value` column contains no valid numeric data for the specified operation (`product-catalog::oteldemo.ProductCatalogService/ListProducts`) in the `apm.operation` entity set. This suggests that there may be no relevant data for this KPI in the dataset.

The original code execution output of IPython Kernel is also provided below for reference:

nan```
```

In [ ]:
# In[4]:

# Filter the metrics data for the specific entity and entity set for the 'error_rate' KPI
filtered_metrics_error_rate = metrics_df[
    (metrics_df['entity_set'] == 'apm.operation') &
    (metrics_df['entity_name'] == 'product-catalog::oteldemo.ProductCatalogService/ListProducts') &
    (metrics_df['metric'] == 'error_rate')
]

# Ensure the 'value' column is numeric for calculations
filtered_metrics_error_rate['value'] = pd.to_numeric(filtered_metrics_error_rate['value'], errors='coerce')

# Calculate the global P95 threshold for the 'error_rate' KPI
p95_threshold_error_rate = filtered_metrics_error_rate['value'].quantile(0.95)

p95_threshold_error_rate

```
Out[4]:
```
The global P95 threshold for the `error_rate` KPI could not be calculated because the `value` column contains no valid numeric data for the specified operation (`product-catalog::oteldemo.ProductCatalogService/ListProducts`) in the `apm.operation` entity set. This indicates that there may be no relevant data for this KPI in the dataset.

The original code execution output of IPython Kernel is also provided below for reference:

nan```
```

In [ ]:
# In[5]:

# Filter the metrics data for the specific service and entity set for the 'workload' KPI
filtered_metrics_service_workload = metrics_df[
    (metrics_df['entity_set'] == 'apm.service.legacy') &
    (metrics_df['entity_name'] == 'product-catalog') &
    (metrics_df['metric'] == 'workload')
]

# Ensure the 'value' column is numeric for calculations
filtered_metrics_service_workload['value'] = pd.to_numeric(filtered_metrics_service_workload['value'], errors='coerce')

# Calculate the global P95 threshold for the 'workload' KPI
p95_threshold_service_workload = filtered_metrics_service_workload['value'].quantile(0.95)

p95_threshold_service_workload

```
Out[5]:
```
The global P95 threshold for the `workload` KPI of the `product-catalog` service is approximately 25,954.6. This value can be used to identify anomalies in the traffic-related metric for the service.

The original code execution output of IPython Kernel is also provided below for reference:

25954.599999999995```
```

In [ ]:
# In[6]:

from datetime import datetime
import pytz

# Define the time window in UTC+8
start_time = datetime.fromisoformat('2026-04-12T21:26:08+08:00').astimezone(pytz.timezone('Asia/Shanghai'))
end_time = datetime.fromisoformat('2026-04-12T21:36:32+08:00').astimezone(pytz.timezone('Asia/Shanghai'))

# Convert the 'time' column to numeric and filter data within the time window
filtered_metrics_service_workload['time'] = pd.to_numeric(filtered_metrics_service_workload['time'], errors='coerce')
time_filtered_data = filtered_metrics_service_workload[
    (filtered_metrics_service_workload['time'] >= start_time.timestamp() * 1_000_000) &
    (filtered_metrics_service_workload['time'] <= end_time.timestamp() * 1_000_000)
]

# Identify anomalies where the 'value' exceeds the global P95 threshold
anomalies = time_filtered_data[time_filtered_data['value'] > p95_threshold_service_workload]

anomalies

```
Out[6]:
```
Within the specified time window (2026-04-12 21:26:08 to 2026-04-12 21:36:32 UTC+8), there were no anomalies detected in the `workload` KPI for the `product-catalog` service, as no values exceeded the global P95 threshold of 25,954.6.

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [time, domain, entity_set, entity_id, entity_name, metric, value, metric_set_id, service]
Index: []```
```

In [ ]:
# In[7]:

# Load the traces.parquet file
traces_file_path = 'rca100/cases/t090/traces.parquet'
traces_df = pd.read_parquet(traces_file_path)

# Ensure relevant columns are numeric for filtering
traces_df['startTime'] = pd.to_numeric(traces_df['startTime'], errors='coerce')
traces_df['duration'] = pd.to_numeric(traces_df['duration'], errors='coerce')
traces_df['statusCode'] = pd.to_numeric(traces_df['statusCode'], errors='coerce')

# Filter traces for the 'product-catalog' service within the specified time window
time_window_start_ns = start_time.timestamp() * 1_000_000_000  # Convert to nanoseconds
time_window_end_ns = end_time.timestamp() * 1_000_000_000  # Convert to nanoseconds

filtered_traces = traces_df[
    (traces_df['serviceName'] == 'product-catalog') &
    (traces_df['startTime'] >= time_window_start_ns) &
    (traces_df['startTime'] <= time_window_end_ns)
]

# Identify traces with high duration or non-zero statusCode
anomalous_traces = filtered_traces[
    (filtered_traces['duration'] > filtered_traces['duration'].quantile(0.95)) |  # High duration
    (filtered_traces['statusCode'] != 0)  # Non-zero statusCode
]

anomalous_traces

```
Out[7]:
```
Within the specified time window (2026-04-12 21:26:08 to 2026-04-12 21:36:32 UTC+8), 466 anomalous traces were identified for the `product-catalog` service. These traces either had high `duration` values (above the 95th percentile) or non-zero `statusCode`, indicating potential issues in the service's operations. For a more detailed analysis, you can request to display additional rows of the data.

The original code execution output of IPython Kernel is also provided below for reference:

traceId            spanId      parentSpanId kind                                     spanName  ...                                             events __topic__      __tag__:__pack_id__    ip links
320883  f8b0dc0926e543bcc01f48cbb8efdfd9  7b12a0de10507169  96432f32a5d4e606    2  oteldemo.ProductCatalogService/ListProducts  ...                                               None            BFC47CCDA6BF0ACA-5789EB  None  None
320888  527ce11b662c385432e6624f922ec3d3  9a3408b74fac3415  be0b4b3736714f6a    2    oteldemo.ProductCatalogService/GetProduct  ...  [{"attributes":{},"name":"Product Found","time...            BFC47CCDA6BF0ACA-5789EB  None  None
321785  54675a26908db8b8608791567d601d93  142fae8ce5d938e3  c59f6e4e583acfa9    2  oteldemo.ProductCatalogService/ListProducts  ...                                               None            BFC47CCDA6BF0ACA-5789EC  None  None
321787  f0d9a07ddcfbad97a8345dd2dd709ad1  a282edd5a0be6160  5274d79bc39ab5ea    2    oteldemo.ProductCatalogService/GetProduct  ...  [{"attributes":{},"name":"Product Found","time...            BFC47CCDA6BF0ACA-5789EC  None  None
326893  712970687afa4ff36e09e9cc486b2074  a0b05363db9b8f56  73ae17263315aafa    2    oteldemo.ProductCatalogService/GetProduct  ...  [{"attributes":{},"name":"Product Found","time...            BFC47CCDA6BF0ACA-5789F3  None  None
...                                  ...               ...               ...  ...                                          ...  ...                                                ...       ...                      ...   ...   ...
506996  f3084a1c8763426dc9cc0a05af6d68f7  044f6865c136f128  8d0153af1cfd3e41    2    oteldemo.ProductCatalogService/GetProduct  ...  [{"attributes":{},"name":"Product Found","time...            9AB0DD0A1D17E183-57663A  None  None
507003  c7568a9ff3355c0d4fa0233bae6cd3f1  8f9932c82069df23  e47292876b0cf93e    2  oteldemo.ProductCatalogService/ListProducts  ...                                               None            9AB0DD0A1D17E183-57663A  None  None
507028  a7a790699c96c68b423c4187f3651aaf  93d8b8c9d1da7d2c  89e4d01fd2741916    2  oteldemo.ProductCatalogService/ListProducts  ...                                               None            9AB0DD0A1D17E183-57663A  None  None
507039  476c57846c212807931808f6e37ecece  798a6b5e3be994cf  bce5f0ec40361923    2  oteldemo.ProductCatalogService/ListProducts  ...                                               None            9AB0DD0A1D17E183-57663A  None  None
507047  51ebfec5285d36270cbaa23e4ba8e159  f6571c7a4f774091  16f3ff7c2556b7db    2    oteldemo.ProductCatalogService/GetProduct  ...  [{"attributes":{},"name":"Product Found","time...            9AB0DD0A1D17E183-57663A  None  None

[466 rows x 21 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[8]:

# Analyze the call chain of the anomalous traces
call_chain_analysis = anomalous_traces[['spanName', 'serviceName', 'statusCode', 'duration']]

# Group by serviceName and spanName to identify the most affected services and operations
call_chain_summary = call_chain_analysis.groupby(['serviceName', 'spanName']).agg(
    count=('spanName', 'size'),
    avg_duration=('duration', 'mean'),
    max_duration=('duration', 'max'),
    error_count=('statusCode', lambda x: (x != 0).sum())
).reset_index()

# Sort by error_count and max_duration to identify the most downstream faulty service
call_chain_summary_sorted = call_chain_summary.sort_values(by=['error_count', 'max_duration'], ascending=False)

call_chain_summary_sorted

```
Out[8]:
```
The analysis of the 466 anomalous traces shows that all issues are confined to the `product-catalog` service. Specifically:

- The `ListProducts` operation was invoked 148 times, with an average duration of 118,062 nanoseconds and a maximum duration of 618,335 nanoseconds. No errors (non-zero `statusCode`) were recorded.
- The `GetProduct` operation was invoked 318 times, with an average duration of 119,381 nanoseconds and a maximum duration of 585,044 nanoseconds. Similarly, no errors were recorded.

No upstream or downstream services of `product-catalog` were identified as experiencing issues, indicating that the anomalies are isolated to the `product-catalog` service's operations.

The original code execution output of IPython Kernel is also provided below for reference:

serviceName                                     spanName  count   avg_duration  max_duration  error_count
1  product-catalog  oteldemo.ProductCatalogService/ListProducts    148  118062.216216        618335            0
0  product-catalog    oteldemo.ProductCatalogService/GetProduct    318  119381.251572        585044            0```
```

In [ ]:
# In[9]:

# Convert the '_time_' column to datetime and make it timezone-aware
product_catalog_logs['_time_'] = pd.to_datetime(product_catalog_logs['_time_'], errors='coerce', unit='s')
product_catalog_logs['_time_'] = product_catalog_logs['_time_'].dt.tz_localize('Asia/Shanghai', ambiguous='NaT', nonexistent='NaT')

# Filter logs within the specified time window
time_filtered_logs = product_catalog_logs[
    (product_catalog_logs['_time_'] >= start_time) &
    (product_catalog_logs['_time_'] <= end_time)
]

# Search for error messages or anomalies in the 'content' field
error_logs = time_filtered_logs[time_filtered_logs['content'].str.contains('error|exception|fail', case=False, na=False)]

error_logs

```
Out[9]:
```
Within the specified time window (2026-04-12 21:26:08 to 2026-04-12 21:36:32 UTC+8), no error messages or anomalies were found in the logs for the `product-catalog` service. This suggests that there were no logged issues or failures during this period.

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [content, _time_, _source_, _image_name_, _container_name_, _pod_name_, _namespace_, _pod_uid_, _container_ip_, __topic__, __tag__:__pack_id__, __tag__:_node_ip_, __tag__:_cluster_id_, __tag__:__hostname__, __tag__:_node_name_]
Index: []

[0 rows x 15 columns]```
```

In [ ]:
# In[10]:

# Filter the metrics data for the specific service and entity set for resource-related KPIs
resource_metrics = metrics_df[
    (metrics_df['entity_set'] == 'apm.service.legacy') &
    (metrics_df['entity_name'] == 'product-catalog') &
    (metrics_df['metric'].isin(['cpu', 'mem']))
]

# Ensure the 'value' column is numeric for calculations
resource_metrics['value'] = pd.to_numeric(resource_metrics['value'], errors='coerce')

# Calculate the global P95 thresholds for 'cpu' and 'mem' KPIs
p95_thresholds_resource = resource_metrics.groupby('metric')['value'].quantile(0.95)

p95_thresholds_resource

```
Out[10]:
```
The global P95 thresholds for resource-related KPIs of the `product-catalog` service are as follows:

- CPU usage: approximately 0.0311 (31.13 mCPU)
- Memory usage: approximately 42.73 MB

These thresholds can be used to identify any anomalies in the resource usage of the `product-catalog` service.

The original code execution output of IPython Kernel is also provided below for reference:

metric
cpu    3.113286e-02
mem    4.273411e+07
Name: value, dtype: float64```
```

In [ ]:
# In[11]:

# Filter the resource metrics for the specific service within the specified time window
time_filtered_resource_metrics = resource_metrics[
    (resource_metrics['time'] >= start_time.timestamp() * 1_000_000) &
    (resource_metrics['time'] <= end_time.timestamp() * 1_000_000)
]

# Identify anomalies where the 'value' exceeds the respective global P95 thresholds
anomalies_resource_usage = time_filtered_resource_metrics[
    ((time_filtered_resource_metrics['metric'] == 'cpu') & (time_filtered_resource_metrics['value'] > 0.0311)) |
    ((time_filtered_resource_metrics['metric'] == 'mem') & (time_filtered_resource_metrics['value'] > 42.73 * 1_000_000))
]

anomalies_resource_usage

```
Out[11]:
```
Within the specified time window (2026-04-12 21:26:08 to 2026-04-12 21:36:32 UTC+8), anomalies were detected in the memory usage (`mem`) of the `product-catalog` service. The memory usage exceeded the global P95 threshold of 42.73 MB at three points, with values ranging from approximately 42.84 MB to 43.25 MB. No anomalies were detected in CPU usage during this period.

The original code execution output of IPython Kernel is also provided below for reference:

time domain          entity_set entity_id      entity_name metric         value metric_set_id service
89742  1776000518000000    apm  apm.service.legacy            product-catalog    mem  4.287966e+07          None    None
90310  1776000758000000    apm  apm.service.legacy            product-catalog    mem  4.284143e+07          None    None
90665  1776000908000000    apm  apm.service.legacy            product-catalog    mem  4.325103e+07          None    None```
```